# Visualise Ancestry 

In [1]:
# visualize_ancestry.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import itertools
import re
import os
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import matplotlib.colors as mcolors
import matplotlib
from typing import List, Tuple, Dict, Any, Optional, Literal # If you want to keep type hints

In [2]:
def get_unique_ancestries(locus_data_df):
    ancestries = set()
    for chromatid in ['chromatid1', 'chromatid2']:
        pop_col = f'ancestry_{chromatid}_pop'
        founder_col = f'ancestry_{chromatid}_founder_id'
        filtered = locus_data_df[[pop_col, founder_col]].dropna()
        for _, row in filtered.iterrows():
            ancestries.add((row[pop_col], int(row[founder_col])))
    return ancestries

def generate_color_map_for_ancestries(unique_ancestries):
    # Use the new matplotlib colormaps API
    cmap = matplotlib.colormaps['tab20'].resampled(len(unique_ancestries))
    color_map = {}
    for i, anc in enumerate(sorted(unique_ancestries)):
        rgba = cmap(i)
        color_map[anc] = mcolors.to_hex(rgba)
    return color_map


In [3]:
def plot_diploid_chromosome_ancestry(
    individual_id: int,
    diploid_chr_id: int,
    generation_label: str,
    locus_data_df: pd.DataFrame,
    marker_height: float = 0.8,
    spacing: float = 1.2,
    save_filename: Optional[str] = None,
    max_loci_per_row: int = 80,
    row_height: float = 3.0
):
    target_data = locus_data_df[
        (locus_data_df['individual_id'] == individual_id) &
        (locus_data_df['diploid_chr_id'] == diploid_chr_id) &
        (locus_data_df['generation'] == generation_label)
    ].sort_values(by='locus_position').reset_index(drop=True)

    if target_data.empty:
        print(f"No data found for Individual ID {individual_id}, Chromosome {diploid_chr_id} in generation {generation_label}.")
        return

    num_loci = len(target_data)
    unique_ancestries = get_unique_ancestries(locus_data_df)
    ancestry_colors = generate_color_map_for_ancestries(unique_ancestries)

    # Determine number of rows to split loci if too many
    num_rows = (num_loci + max_loci_per_row - 1) // max_loci_per_row
    loci_per_row = min(num_loci, max_loci_per_row)

    fig_width = max(12, loci_per_row * 0.25)  # increase spacing per locus a bit
    fig_height = row_height * num_rows

    fig, axs = plt.subplots(num_rows, 1, figsize=(fig_width, fig_height), squeeze=False)
    axs = axs.flatten()

    legend_elements = []
    used_ancestries = set()

    for row_idx in range(num_rows):
        ax = axs[row_idx]
        start_idx = row_idx * max_loci_per_row
        end_idx = min(start_idx + max_loci_per_row, num_loci)
        row_data = target_data.iloc[start_idx:end_idx]

        ax.set_title(f"Individual {individual_id}, Chromosome {diploid_chr_id} ({generation_label})\n"
                     "Ancestral Origin", fontsize=14 if row_idx == 0 else 10)
        ax.set_xlabel("Locus Position", fontsize=12)
        ax.set_ylabel("Chromatid", fontsize=12)
        ax.set_yticks([-spacing/2, spacing/2])
        ax.set_yticklabels(['Chromatid 2', 'Chromatid 1'])

        tick_interval = max(1, loci_per_row // 20)
        ticks = np.arange(0, loci_per_row, tick_interval)
        labels = ticks + start_idx  # shift labels to match loci indices

        ax.set_xticks(ticks)
        ax.set_xticklabels(labels)

        ax.set_xlim(-0.5, loci_per_row - 0.5)
        ax.set_ylim(-spacing, spacing)
        ax.set_aspect('auto', adjustable='box')
        ax.axis('off')

        ax.plot([-0.5, loci_per_row - 0.5], [spacing/2, spacing/2], color='black', linewidth=1.5, zorder=1)
        ax.plot([-0.5, loci_per_row - 0.5], [-spacing/2, -spacing/2], color='black', linewidth=1.5, zorder=1)

        for i, row in row_data.iterrows():
            local_idx = i - start_idx
            for chromatid, y_offset in zip(['chromatid1', 'chromatid2'], [spacing/2, -spacing/2]):
                anc_pop = row[f'ancestry_{chromatid}_pop']
                founder_id = row[f'ancestry_{chromatid}_founder_id']
                if pd.isna(anc_pop) or pd.isna(founder_id):
                    continue
                ancestry = (anc_pop, int(founder_id))
                color = ancestry_colors.get(ancestry, 'gray')

                rect = patches.Rectangle(
                    (local_idx - 0.4, y_offset - marker_height/2),
                    0.8,
                    marker_height,
                    facecolor=color,
                    edgecolor='none',
                    linewidth=0,
                    zorder=2
                )
                ax.add_patch(rect)

                # Calculate text color contrast
                try:
                    import matplotlib.colors as mcolors
                    rgb = mcolors.to_rgb(color)
                    text_color = 'black' if np.mean(rgb) > 0.5 else 'white'
                except Exception:
                    text_color = 'black'

                ax.text(
                    local_idx,
                    y_offset,
                    str(founder_id),
                    ha='center',
                    va='center',
                    fontsize=7,
                    color=text_color,
                    zorder=3
                )

                if ancestry not in used_ancestries:
                    legend_elements.append(Line2D([0], [0], marker='s', color='w', label=f'{anc_pop}_{founder_id}',
                                                  markerfacecolor=color, markersize=8))
                    used_ancestries.add(ancestry)

    if legend_elements:
        axs[-1].legend(
            handles=legend_elements,
            title="Ancestral Origin",
            loc='upper right',
            bbox_to_anchor=(1.05, 1),
            borderaxespad=0.0,
            frameon=False,
            fontsize=9,
            title_fontsize=10
        )

    fig.subplots_adjust(right=0.8, hspace=0.4)

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Chromosome plot saved to: {save_filename}")
        plt.close(fig)  # Close to free memory
    else:
        return fig

In [4]:
# Example 1: List IDs in a specific generation
import pandas as pd

# Load the locus-level genotype data with ancestry info
locus_level_df = pd.read_csv('output_data/dataframes/locus_level_genotypes_with_ancestry.csv')

generation_to_check = 'F5'  # Change this to any generation you want to inspect
print(f"\n--- Individual IDs in Generation '{generation_to_check}' ---")
ids_in_generation = locus_level_df.loc[
    locus_level_df['generation'] == generation_to_check,
    'individual_id'
].unique().tolist()

if ids_in_generation:
    print(f"Total individuals in '{generation_to_check}': {len(ids_in_generation)}")
    print("First 10 IDs (if available):", ids_in_generation[:10])
    print("Last 10 IDs (if available):", ids_in_generation[-10:])
else:
    print(f"No individuals found for generation '{generation_to_check}'.")


--- Individual IDs in Generation 'F5' ---
Total individuals in 'F5': 100
First 10 IDs (if available): [600, 601, 602, 603, 604, 605, 606, 607, 608, 609]
Last 10 IDs (if available): [690, 691, 692, 693, 694, 695, 696, 697, 698, 699]


In [5]:
if __name__ == "__main__":
    print("\nStarting Chr Ancestry Visualisation")

    # Plot a SPECIFIC individual by its ID
    # NOTE: Make sure this ID actually exists in your data for the specified generation!
    # You might get this ID from the list printed in Example 1, or from other analysis.
    specific_id_to_plot = 602  # <--- CHANGE THIS to an ID you know exists!
    specific_gen_for_id = 'F5'  # <--- CHANGE THIS to the generation of that specific ID!
    specific_chr_to_plot = 2  # <--- CHANGE THIS to the chromosome pair (1 or 2)

    plots_directory = 'output_data/plots'  # or wherever you want

    # Make sure the directory exists
    os.makedirs(plots_directory, exist_ok=True)

    # Check if the specific individual exists in the DataFrame before attempting to plot
    if not locus_level_df[
        (locus_level_df['individual_id'] == specific_id_to_plot) &
        (locus_level_df['generation'] == specific_gen_for_id) &
        (locus_level_df['diploid_chr_id'] == specific_chr_to_plot)
    ].empty:

        plot_save_path_specific_ind = os.path.join(
            plots_directory,
            f"specific_ind_{specific_id_to_plot}_gen_{specific_gen_for_id}_chr_{specific_chr_to_plot}_ancestry.png"
        )

        print(f"\nPlotting specific individual (ID: {specific_id_to_plot}) from generation '{specific_gen_for_id}', Chromosome {specific_chr_to_plot}")
        plot_diploid_chromosome_ancestry(
            individual_id=specific_id_to_plot,
            diploid_chr_id=specific_chr_to_plot,
            generation_label=specific_gen_for_id,
            locus_data_df=locus_level_df,
            save_filename=plot_save_path_specific_ind
        )
    else:
        print(f"\nSkipping plot for specific individual {specific_id_to_plot}: Data not found for this ID, generation, and chromosome combination.")



Starting Chr Ancestry Visualisation

Plotting specific individual (ID: 602) from generation 'F5', Chromosome 2
Chromosome plot saved to: output_data/plots\specific_ind_602_gen_F5_chr_2_ancestry.png


### FUll SCRIPT dup 

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import numpy as np
import os
from typing import Optional, List, Dict, Tuple
from matplotlib.lines import Line2D # For legend elements

# --- Existing Helper Functions (unchanged) ---
def get_unique_ancestries(locus_data_df):
    ancestries = set()
    for chromatid in ['chromatid1', 'chromatid2']:
        pop_col = f'ancestry_{chromatid}_pop'
        founder_col = f'ancestry_{chromatid}_founder_id'
        filtered = locus_data_df[[pop_col, founder_col]].dropna()
        for _, row in filtered.iterrows():
            ancestries.add((row[pop_col], int(row[founder_col])))
    return ancestries

def generate_color_map_for_ancestries(unique_ancestries):
    cmap = matplotlib.colormaps['tab20'].resampled(len(unique_ancestries))
    color_map = {}
    for i, anc in enumerate(sorted(unique_ancestries)):
        rgba = cmap(i)
        color_map[anc] = mcolors.to_hex(rgba)
    return color_map

def consolidate_ancestry_blocks(locus_data: pd.DataFrame, chromatid_label: str):
    blocks = []
    current_block_ancestry = None
    current_block_start_local_idx = 0

    pop_col = f'ancestry_{chromatid_label}_pop'
    founder_col = f'ancestry_{chromatid_label}_founder_id'

    if locus_data.empty:
        return []

    original_locus_positions = locus_data['locus_position'].tolist()

    for i, row in locus_data.iterrows():
        anc_pop = row[pop_col]
        founder_id = row[founder_col]

        if pd.isna(anc_pop) or pd.isna(founder_id):
            current_ancestry = (None, None)
        else:
            current_ancestry = (anc_pop, int(founder_id))

        if current_ancestry != current_block_ancestry:
            if current_block_ancestry is not None:
                blocks.append({
                    'start_idx': current_block_start_local_idx,
                    'end_idx': i - 1,
                    'ancestry': current_block_ancestry
                })

            current_block_ancestry = current_ancestry
            current_block_start_local_idx = i

    if current_block_ancestry is not None:
        blocks.append({
            'start_idx': current_block_start_local_idx,
            'end_idx': len(locus_data) - 1,
            'ancestry': current_block_ancestry
        })

    return blocks

'''def plot_diploid_chromosome_ancestry(
    individual_id: int,
    diploid_chr_id: int,
    generation_label: str,
    locus_data_df: pd.DataFrame,
    marker_height: float = 0.8,
    spacing: float = 1.2,
    save_filename: Optional[str] = None,
    max_loci_per_row: int = 80,
    row_height: float = 3.0
):
    target_data = locus_data_df[
        (locus_data_df['individual_id'] == individual_id) &
        (locus_data_df['diploid_chr_id'] == diploid_chr_id) &
        (locus_data_df['generation'] == generation_label)
    ].sort_values(by='locus_position').reset_index(drop=True)

    if target_data.empty:
        print(f"No data found for Individual ID {individual_id}, Chromosome {diploid_chr_id} in generation {generation_label}.")
        return

    num_loci = len(target_data)

    unique_ancestries = get_unique_ancestries(locus_data_df)
    ancestry_colors = generate_color_map_for_ancestries(unique_ancestries)

    if (None, None) not in ancestry_colors:
        ancestry_colors[(None, None)] = 'gray'

    num_rows = (num_loci + max_loci_per_row - 1) // max_loci_per_row
    loci_per_row = min(num_loci, max_loci_per_row)

    fig_width = max(12, loci_per_row * 0.2)
    fig_height = row_height * num_rows

    fig, axs = plt.subplots(num_rows, 1, figsize=(fig_width, fig_height), squeeze=False)
    axs = axs.flatten()

    legend_elements = []
    used_ancestries = set()

    for row_idx in range(num_rows):
        ax = axs[row_idx]
        start_locus_idx_for_row = row_idx * max_loci_per_row
        end_locus_idx_for_row = min(start_locus_idx_for_row + max_loci_per_row, num_loci)
        row_data = target_data.iloc[start_locus_idx_for_row:end_locus_idx_for_row].copy().reset_index(drop=True)

        ax.set_title(f"Individual {individual_id}, Chromosome {diploid_chr_id} ({generation_label})\n"
                     "Ancestral Origin", fontsize=14 if row_idx == 0 else 10)
        ax.set_xlabel("Locus Index (relative to start of chromosome)", fontsize=12)
        ax.set_ylabel("Chromatid", fontsize=12)
        ax.set_yticks([-spacing/2, spacing/2])
        ax.set_yticklabels(['Chromatid 2', 'Chromatid 1'])

        tick_interval = max(1, loci_per_row // 20)
        ticks = np.arange(0, loci_per_row, tick_interval)
        labels = ticks + start_locus_idx_for_row

        ax.set_xticks(ticks)
        ax.set_xticklabels(labels)

        ax.set_xlim(-0.5, loci_per_row - 0.5)
        ax.set_ylim(-spacing, spacing)
        ax.set_aspect('auto', adjustable='box')
        ax.axis('off')

        ax.plot([-0.5, loci_per_row - 0.5], [spacing/2, spacing/2], color='black', linewidth=1.5, zorder=1)
        ax.plot([-0.5, loci_per_row - 0.5], [-spacing/2, -spacing/2], color='black', linewidth=1.5, zorder=1)

        ax.text(-0.7, spacing/2, 'Chromatid 1', va='center', ha='right', fontsize=10, transform=ax.get_yaxis_transform(0))
        ax.text(-0.7, -spacing/2, 'Chromatid 2', va='center', ha='right', fontsize=10, transform=ax.get_yaxis_transform(0))

        for chromatid, y_offset in zip(['chromatid1', 'chromatid2'], [spacing/2, -spacing/2]):
            blocks = consolidate_ancestry_blocks(row_data, chromatid)

            for block in blocks:
                block_start_local_idx = block['start_idx']
                block_end_local_idx = block['end_idx']
                ancestry = block['ancestry']

                if ancestry == (None, None):
                    display_label = "Missing Data"
                    color = ancestry_colors.get((None, None), 'gray')
                else:
                    anc_pop, founder_id = ancestry
                    display_label = f'{anc_pop}_{founder_id}'
                    color = ancestry_colors.get(ancestry, 'gray')

                rect_width = (block_end_local_idx - block_start_local_idx + 1) * 0.8
                rect_x = block_start_local_idx - 0.4

                rect = patches.Rectangle(
                    (rect_x, y_offset - marker_height/2),
                    rect_width,
                    marker_height,
                    facecolor=color,
                    edgecolor='none',
                    linewidth=0,
                    zorder=2
                )
                ax.add_patch(rect)

                if ancestry != (None, None):
                    text_x_pos = block_start_local_idx + (block_end_local_idx - block_start_local_idx) / 2

                    try:
                        rgb = mcolors.to_rgb(color)
                        text_color = 'black' if np.mean(rgb) > 0.5 else 'white'
                    except Exception:
                        text_color = 'black'

                    ax.text(
                        text_x_pos,
                        y_offset,
                        str(founder_id),
                        ha='center',
                        va='center',
                        fontsize=7,
                        color=text_color,
                        zorder=3
                    )

                if display_label not in used_ancestries and ancestry != (None, None):
                    legend_elements.append(Line2D([0], [0], marker='s', color='w', label=display_label,
                                                  markerfacecolor=color, markersize=8))
                    used_ancestries.add(display_label)
                elif ancestry == (None, None) and "Missing Data" not in used_ancestries:
                    legend_elements.append(Line2D([0], [0], marker='s', color='w', label="Missing Data",
                                                  markerfacecolor='gray', markersize=8))
                    used_ancestries.add("Missing Data")

    if legend_elements:
        legend_elements.sort(key=lambda x: x.get_label())

        fig.legend(
            handles=legend_elements,
            title="Ancestral Origin",
            loc='center right',
            bbox_to_anchor=(0.98, 0.5),
            borderaxespad=0.0,
            frameon=False,
            fontsize=9,
            title_fontsize=10
        )

    fig.subplots_adjust(right=0.75, hspace=0.4)

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Chromosome plot saved to: {save_filename}")
        plt.close(fig)
    else:
        plt.show()
        return fig '''

def generate_ancestry_block_summary_df(locus_data_df: pd.DataFrame) -> pd.DataFrame:
    summary_data = []

    unique_ids_chrs_gens = locus_data_df[['individual_id', 'diploid_chr_id', 'generation']].drop_duplicates()

    for _, row_id_chr_gen in unique_ids_chrs_gens.iterrows():
        ind_id = row_id_chr_gen['individual_id']
        chr_id = row_id_chr_gen['diploid_chr_id']
        gen_label = row_id_chr_gen['generation']

        target_data = locus_data_df[
            (locus_data_df['individual_id'] == ind_id) &
            (locus_data_df['diploid_chr_id'] == chr_id) &
            (locus_data_df['generation'] == gen_label)
        ].sort_values(by='locus_position').reset_index(drop=True)

        if target_data.empty:
            continue

        chromatid1_blocks_summary = []
        chromatid2_blocks_summary = []

        blocks1 = consolidate_ancestry_blocks(target_data, 'chromatid1')
        for block in blocks1:
            start_locus = target_data.loc[block['start_idx'], 'locus_position']
            end_locus = target_data.loc[block['end_idx'], 'locus_position']
            anc = block['ancestry']
            if anc == (None, None):
                chromatid1_blocks_summary.append(f"{start_locus}-{end_locus}: Missing")
            else:
                chromatid1_blocks_summary.append(f"{start_locus}-{end_locus}: {anc[0]}{anc[1]}")

        blocks2 = consolidate_ancestry_blocks(target_data, 'chromatid2')
        for block in blocks2:
            start_locus = target_data.loc[block['start_idx'], 'locus_position']
            end_locus = target_data.loc[block['end_idx'], 'locus_position']
            anc = block['ancestry']
            if anc == (None, None):
                chromatid2_blocks_summary.append(f"{start_locus}-{end_locus}: Missing")
            else:
                chromatid2_blocks_summary.append(f"{start_locus}-{end_locus}: {anc[0]}{anc[1]}")

        summary_data.append({
            'individual_id': ind_id,
            'diploid_chr_id': chr_id,
            'generation': gen_label,
            'chromatid1_ancestry_blocks': chromatid1_blocks_summary,
            'chromatid2_ancestry_blocks': chromatid2_blocks_summary,
            'num_blocks_chromatid1': len(blocks1), # Add block counts
            'num_blocks_chromatid2': len(blocks2) # Add block counts
        })

    return pd.DataFrame(summary_data)

# --- NEW FUNCTION FOR LINE GRAPH ---
def plot_blocks_per_generation(
    ancestry_summary_df: pd.DataFrame,
    save_filename: Optional[str] = None
):
    """
    Plots the average number of ancestry blocks per chromatid against generation time.

    Args:
        ancestry_summary_df (pd.DataFrame): The DataFrame generated by generate_ancestry_block_summary_df.
        save_filename (Optional[str]): Path to save the plot. If None, displays the plot.
    """
    if ancestry_summary_df.empty:
        print("Ancestry summary DataFrame is empty. Cannot generate blocks per generation plot.")
        return

    # Calculate total blocks per individual (sum of blocks on both chromatids)
    # And then calculate blocks per chromatid
    summary_df_copy = ancestry_summary_df.copy() # Avoid modifying original
    summary_df_copy['total_blocks'] = summary_df_copy['num_blocks_chromatid1'] + summary_df_copy['num_blocks_chromatid2']
    summary_df_copy['blocks_per_chromatid'] = summary_df_copy['total_blocks'] / 2

    # Group by generation and calculate the mean number of blocks per chromatid
    # We need to handle generation labels that might be strings like 'F1', 'BC1', 'F2', 'BC1A' etc.
    # For a line graph, we usually want a numerical progression.
    # If generations are 'F1', 'F2', 'F3', etc., we can sort them.
    # If they include 'BC' generations, custom sorting might be needed or numerical mapping.
    
    # Attempt to convert generations to numeric for sorting if possible, otherwise use string sort.
    # This assumes 'F' generations are F1, F2, F3... and 'BC' generations are BC1, BC2...
    # For complex labels like 'BC1A', 'BC1B', etc., they will be treated alphabetically within 'BC1'.
    
    def try_numeric_generation(gen_label):
        if isinstance(gen_label, str) and gen_label.startswith('F'):
            try:
                return int(gen_label[1:]) # F1 -> 1, F2 -> 2
            except ValueError:
                return gen_label # Keep original if not simple Fx
        elif isinstance(gen_label, str) and gen_label.startswith('BC'):
            try:
                # Handle BC1, BC2. For BC1A, BC1B, etc., it will still sort alphabetically.
                return 'BC' + str(int(''.join(filter(str.isdigit, gen_label)))) + gen_label.strip('0123456789')
            except ValueError:
                return gen_label
        return gen_label # Keep original for others

    # Create a sortable key for generations
    summary_df_copy['generation_sort_key'] = summary_df_copy['generation'].apply(try_numeric_generation)

    # Group and sort
    blocks_per_gen = summary_df_copy.groupby('generation_sort_key')['blocks_per_chromatid'].mean().reset_index()
    
    # Re-map sort_key back to original generation label for display if needed, or just use original 'generation'
    # For plotting, it's better to use the original string labels for the ticks, but sort by the key.
    
    # Map back original generation labels for plotting, ensuring correct order
    # Get original generation labels corresponding to the sorted keys
    generation_labels_sorted = summary_df_copy[['generation_sort_key', 'generation']].drop_duplicates().sort_values('generation_sort_key')
    
    # Merge to ensure correct order of display labels
    blocks_per_gen = pd.merge(blocks_per_gen, generation_labels_sorted, on='generation_sort_key', how='left').drop_duplicates().sort_values('generation_sort_key')


    if blocks_per_gen.empty:
        print("No aggregated data to plot for blocks per generation.")
        return

    plt.figure(figsize=(10, 6))
    plt.plot(blocks_per_gen['generation'], blocks_per_gen['blocks_per_chromatid'], marker='o', linestyle='-', color='skyblue')

    plt.title('Average Number of Ancestry Blocks Per Chromatid Over Generations', fontsize=16)
    plt.xlabel('Generation', fontsize=12)
    plt.ylabel('Average Number of Blocks', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels if they are long
    plt.tight_layout() # Adjust layout to prevent labels overlapping

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Blocks per generation plot saved to: {save_filename}")
        plt.close()
    else:
        plt.show()

# --- Main execution block (updated to include new plot) ---
if __name__ == "__main__":
    print("\nStarting Chromosome Ancestry Analysis: Visualization, Summary, and Block Count Plot")

    # Load the locus-level genotype data with ancestry info
    # Make sure this path is correct for your setup
    locus_level_df = pd.read_csv('output_data/dataframes/locus_level_genotypes_with_ancestry.csv')

    # Example 1: List IDs in a specific generation
    generation_to_check = 'F5'
    print(f"\n--- Individual IDs in Generation '{generation_to_check}' ---")
    ids_in_generation = locus_level_df.loc[
        locus_level_df['generation'] == generation_to_check,
        'individual_id'
    ].unique().tolist()

    if ids_in_generation:
        print(f"Total individuals in '{generation_to_check}': {len(ids_in_generation)}")
        print("First 10 IDs (if available):", ids_in_generation[:10])
        print("Last 10 IDs (if available):", ids_in_generation[-10:])
    else:
        print(f"No individuals found for generation '{generation_to_check}'.")

    # --- Generate the supplementary DataFrame ---
    print("\nGenerating Ancestry Block Summary DataFrame...")
    # Pass a copy to generate_ancestry_block_summary_df to avoid potential SettingWithCopyWarning
    ancestry_summary_df = generate_ancestry_block_summary_df(locus_level_df.copy())

    # Save the summary DataFrame
    summary_output_path = 'output_data/dataframes/ancestry_block_summary.csv'
    os.makedirs(os.path.dirname(summary_output_path), exist_ok=True)
    ancestry_summary_df.to_csv(summary_output_path, index=False)
    print(f"Ancestry block summary saved to: {summary_output_path}")
    print("\nFirst 5 rows of the Ancestry Block Summary (including new block counts):")
    print(ancestry_summary_df.head())
    print("\n--- End of Ancestry Block Summary ---")

    # --- Plotting Average Blocks Per Generation ---
    blocks_per_gen_plot_path = os.path.join(plots_directory, "average_ancestry_blocks_per_generation.png")
    print(f"\nPlotting average number of ancestry blocks per generation to: {blocks_per_gen_plot_path}")
    plot_blocks_per_generation(ancestry_summary_df, save_filename=blocks_per_gen_plot_path)


    # --- Plotting a SPECIFIC individual (same as before) ---
    specific_id_to_plot = 121
    specific_gen_for_id = 'F5'
    specific_chr_to_plot = 2

    plots_directory = 'output_data/plots'
    os.makedirs(plots_directory, exist_ok=True)

    if not locus_level_df[
        (locus_level_df['individual_id'] == specific_id_to_plot) &
        (locus_level_df['generation'] == specific_gen_for_id) &
        (locus_level_df['diploid_chr_id'] == specific_chr_to_plot)
    ].empty:

        plot_save_path_specific_ind = os.path.join(
            plots_directory,
            f"specific_ind_{specific_id_to_plot}_gen_{specific_gen_for_id}_chr_{specific_chr_to_plot}_ancestry_merged.png"
        )

        print(f"\nPlotting specific individual (ID: {specific_id_to_plot}) from generation '{specific_gen_for_id}', Chromosome {specific_chr_to_plot} with merged blocks.")
        plot_diploid_chromosome_ancestry(
            individual_id=specific_id_to_plot,
            diploid_chr_id=specific_chr_to_plot,
            generation_label=specific_gen_for_id,
            locus_data_df=locus_level_df,
            save_filename=plot_save_path_specific_ind
        )
    else:
        print(f"\nSkipping plot for specific individual {specific_id_to_plot}: Data not found for this ID, generation, and chromosome combination.")


Starting Chromosome Ancestry Analysis: Visualization, Summary, and Block Count Plot

--- Individual IDs in Generation 'F5' ---
Total individuals in 'F5': 20
First 10 IDs (if available): [120, 121, 122, 123, 124, 125, 126, 127, 128, 129]
Last 10 IDs (if available): [130, 131, 132, 133, 134, 135, 136, 137, 138, 139]

Generating Ancestry Block Summary DataFrame...
Ancestry block summary saved to: output_data/dataframes/ancestry_block_summary.csv

First 5 rows of the Ancestry Block Summary (including new block counts):
   individual_id  diploid_chr_id generation chromatid1_ancestry_blocks  \
0              0               1        P_A               [0-99: P_A0]   
1              0               2        P_A               [0-99: P_A0]   
2              1               1        P_A               [0-99: P_A1]   
3              1               2        P_A               [0-99: P_A1]   
4              2               1        P_A               [0-99: P_A2]   

  chromatid2_ancestry_blocks  num

NameError: name 'plots_directory' is not defined